# ⚡ Qwen3 SQL Analyst
**Ask questions about your Excel/CSV in plain English → Qwen3 generates & runs SQL**

| Cell | What it does |
|------|--------------|
| **Cell 1** | Install packages |
| **Cell 2** | Choose model & load Qwen3 |
| **Cell 3** | Upload your Excel or CSV file |
| **Cell 4** | Inspect schema (columns, types, samples) |
| **Cell 5** | Ask questions — agent generates & runs SQL |

> 💡 **Runtime tip:** Go to `Runtime → Change runtime type → T4 GPU` before starting.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install Packages
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas openpyxl xlrd

import torch
print(f'✅ Packages ready')
print(f'🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU found — switch runtime to T4)"}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.3 MB/s eta 0:00:00
✅ Packages ready
🖥️  GPU: Tesla T4
💾 VRAM: 15.6 GB


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Choose Model & Load Qwen3
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  MODEL OPTIONS:
#    "Qwen/Qwen3-4B"   →  ~4 GB VRAM  — fast, works on free T4
#    "Qwen/Qwen3-8B"   →  ~8 GB VRAM  — smarter, fits T4 with 4-bit
#
MODEL_NAME = "Qwen/Qwen3-4B"   # ← change here
USE_4BIT   = False               # 4-bit quant: saves VRAM, minimal quality loss

# ─────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print(f'🔄 Loading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if USE_4BIT and torch.cuda.is_available():
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4'
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_cfg,
        device_map='auto',
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
        trust_remote_code=True
    )

model.eval()
print(f'✅ {MODEL_NAME} loaded and ready!')

🔄 Loading Qwen/Qwen3-4B ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Qwen/Qwen3-4B loaded and ready!


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Upload Your File  (Excel or CSV)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
import sqlite3
import re
from google.colab import files

print('📂 Select your Excel (.xlsx / .xls) or CSV file:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'\n✅ Received: {filename}')

# ── Load into DataFrame ──────────────────────────────
if filename.lower().endswith('.csv'):
    df = pd.read_csv(filename)
else:
    xf = pd.ExcelFile(filename)
    if len(xf.sheet_names) > 1:
        print(f'📋 Sheets found: {xf.sheet_names}')
        sheet = input(f'Enter sheet name to load (default: {xf.sheet_names[0]}): ').strip()
        sheet = sheet if sheet in xf.sheet_names else xf.sheet_names[0]
    else:
        sheet = xf.sheet_names[0]
    df = pd.read_excel(filename, sheet_name=sheet)
    print(f'📄 Loaded sheet: "{sheet}"')

# ── Clean column names for SQL compatibility ─────────
original_cols = list(df.columns)
df.columns = [
    re.sub(r'[^\w]', '_', str(c)).strip('_').lower() or f'col_{i}'
    for i, c in enumerate(df.columns)
]
df.columns = [f'c_{c}' if c[0].isdigit() else c for c in df.columns]

# ── Load into in-memory SQLite ───────────────────────
TABLE = 'data'
conn  = sqlite3.connect(':memory:')
df.to_sql(TABLE, conn, if_exists='replace', index=False)

print(f'\n📊 Loaded {len(df):,} rows × {len(df.columns)} columns into SQLite table "{TABLE}"')
print('   → Run Cell 4 to inspect the schema')

📂 Select your Excel (.xlsx / .xls) or CSV file:


Saving Updated Quality of Life Data.csv to Updated Quality of Life Data.csv

✅ Received: Updated Quality of Life Data.csv

📊 Loaded 10,000 rows × 8 columns into SQLite table "data"
   → Run Cell 4 to inspect the schema


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Inspect Schema  (runs automatically, read-only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
import numpy as np

def build_schema_context(df: pd.DataFrame, table: str, conn, n_samples: int = 3) -> str:
    """
    Builds a rich schema string that is injected into the Qwen3 prompt.
    Includes: column names, SQL types, pandas dtypes, null %, unique count,
    min/max (numeric), and sample values.
    """
    lines = []
    lines.append(f'TABLE NAME : {table}')
    lines.append(f'TOTAL ROWS : {len(df):,}')
    lines.append(f'TOTAL COLS : {len(df.columns)}')
    lines.append('')
    lines.append(f'{"COLUMN":<28} {"SQL TYPE":<12} {"PD DTYPE":<12} {"NULL%":>6}  {"UNIQUE":>7}  RANGE / SAMPLES')
    lines.append('─' * 100)

    # Get SQLite column types
    pragma = conn.execute(f'PRAGMA table_info({table})').fetchall()
    sql_types = {row[1]: row[2] for row in pragma}

    for col in df.columns:
        series     = df[col]
        sql_type   = sql_types.get(col, 'TEXT')
        pd_dtype   = str(series.dtype)
        null_pct   = f"{series.isna().mean()*100:.1f}%"
        n_unique   = series.nunique()

        # Range for numerics, sample values for strings/dates
        if pd.api.types.is_numeric_dtype(series):
            mn = series.min()
            mx = series.max()
            detail = f'min={mn:,.2f}  max={mx:,.2f}'
        else:
            samples = series.dropna().unique()[:n_samples]
            samples = [str(s)[:20] for s in samples]
            detail  = 'e.g. ' + ' | '.join(f'"{s}"' for s in samples)

        lines.append(f'{col:<28} {sql_type:<12} {pd_dtype:<12} {null_pct:>6}  {n_unique:>7}  {detail}')

    lines.append('')
    lines.append(f'SAMPLE ROWS (first {n_samples}):')
    lines.append(df.head(n_samples).to_string(index=False))
    return '\n'.join(lines)


# Build and store the schema context (used in Cell 5)
SCHEMA_CONTEXT = build_schema_context(df, TABLE, conn)

print('╔' + '═'*70 + '╗')
print('║  SCHEMA INSPECTOR                                                    ║')
print('╚' + '═'*70 + '╝')
print(SCHEMA_CONTEXT)

╔══════════════════════════════════════════════════════════════════════╗
║  SCHEMA INSPECTOR                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
TABLE NAME : data
TOTAL ROWS : 10,000
TOTAL COLS : 8

COLUMN                       SQL TYPE     PD DTYPE      NULL%   UNIQUE  RANGE / SAMPLES
────────────────────────────────────────────────────────────────────────────────────────────────────
id                           INTEGER      int64          0.0%    10000  min=10,001.00  max=20,000.00
gender                       TEXT         object         0.0%        2  e.g. "Female" | "Male"
occupation_type              TEXT         object         0.0%       14  e.g. "Teacher" | "Office Worker" | "Manager"
avg_work_hours_per_day       REAL         float64        0.0%     1406  min=0.01  max=23.97
avg_rest_hours_per_day       REAL         float64        0.0%     1328  min=0.00  max=23.93
avg_sleep_hours_per_day      REAL        

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — SQL Agent  (schema-aware Qwen3 + auto-retry)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re, textwrap
import pandas as pd
import torch
from IPython.display import display

# ── 1. Qwen3 inference helper ────────────────────────
def call_qwen(prompt: str, max_new_tokens: int = 512, temperature: float = 0.1) -> str:
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an expert SQL analyst. Your job is to write correct, '
                'efficient SQLite queries. Always wrap SQL in ```sql ... ``` blocks. '
                'After the SQL block, write one short sentence explaining what it does.'
            )
        },
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()


# ── 2. Extract SQL from model output ────────────────
def extract_sql(text: str) -> str:
    # Try ```sql ... ```
    m = re.search(r'```sql\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if m: return m.group(1).strip()
    # Try bare ``` ... ```
    m = re.search(r'```\s*(SELECT.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if m: return m.group(1).strip()
    # Try raw SELECT statement
    m = re.search(r'(SELECT[\s\S]+?)(?:\n{2,}|$)', text, re.IGNORECASE)
    if m: return m.group(1).strip()
    return ''


# ── 3. Execute SQL safely ────────────────────────────
def run_sql(sql: str):
    sql_clean = sql.rstrip(';').strip()
    try:
        result = pd.read_sql_query(sql_clean, conn)
        return result, None
    except Exception as e:
        return None, str(e)


# ── 4. Build the schema-aware prompt ─────────────────
def build_prompt(question: str, schema: str, error: str = None) -> str:
    base = textwrap.dedent(f"""
        You have access to a SQLite database with the following table.
        Study the schema carefully — use the exact column names, respect the types,
        and account for null values when needed.

        ─── SCHEMA ───────────────────────────────────────────
        {schema}
        ──────────────────────────────────────────────────────

        QUESTION: {question}

        RULES:
        - Use only SQLite-compatible SQL
        - Wrap SQL in ```sql ... ``` code blocks
        - Use exact column names from the schema above
        - Add LIMIT 500 unless a specific count is requested
        - Use IS NULL / IS NOT NULL for null checks
        - After the SQL block write one brief explanation sentence
    """).strip()

    if error:
        base += f'\n\nThe previous attempt produced this error:\n  {error}\nPlease fix it.'

    return base


# ── 5. Pretty-print results ───────────────────────────
def display_results(question, sql, explanation, result_df, attempt):
    sep = '─' * 72
    print(f'\n{sep}')
    print(f'  ❓ QUESTION   : {question}')
    print(f'  🔁 ATTEMPT    : {attempt}')
    print(sep)
    print('  📝 GENERATED SQL:')
    for line in sql.splitlines():
        print(f'     {line}')
    if explanation:
        clean_expl = re.sub(r'```[\s\S]*?```', '', explanation).strip()
        if clean_expl:
            print(f'\n  💡 EXPLANATION: {clean_expl[:200]}')
    print(f'\n  📊 RESULTS  ({len(result_df):,} rows × {len(result_df.columns)} cols):')
    print(sep)
    display(result_df)


# ── 6. Main agent function ────────────────────────────
def sql_agent(question: str, max_retries: int = 3, verbose: bool = True):
    """
    Schema-aware SQL agent:
      1. Retrieves full schema (column names, types, nulls, ranges, samples)
      2. Builds a rich prompt and calls Qwen3
      3. Extracts and executes the generated SQL
      4. Auto-retries up to max_retries times on SQL errors
    """
    error = None

    for attempt in range(1, max_retries + 1):
        if verbose:
            print(f'🤖 Attempt {attempt}/{max_retries} — calling Qwen3...', end='', flush=True)

        # Build prompt (includes error feedback on retries)
        prompt   = build_prompt(question, SCHEMA_CONTEXT, error)
        response = call_qwen(prompt)

        if verbose:
            print(' done.')

        # Extract SQL
        sql = extract_sql(response)
        if not sql:
            error = 'Could not find a SQL block in the model output.'
            if verbose: print(f'  ⚠️  {error}')
            continue

        # Run SQL
        result_df, sql_error = run_sql(sql)

        if sql_error:
            error = sql_error
            if verbose: print(f'  ⚠️  SQL error: {sql_error}')
            continue

        # Success!
        explanation = re.sub(r'```[\s\S]*?```', '', response).strip()
        display_results(question, sql, explanation, result_df, attempt)
        return result_df

    print(f'\n❌ Failed after {max_retries} attempts. Last error: {error}')
    return None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ✏️  TYPE YOUR QUESTION HERE and run the cell
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
QUESTION = input("Enter Your Question: ")  # ← change this

result = sql_agent(QUESTION)

Enter Your Question: what occupation has the highst amount of sleep hours? sort from high to low
🤖 Attempt 1/3 — calling Qwen3... done.

────────────────────────────────────────────────────────────────────────
  ❓ QUESTION   : what occupation has the highst amount of sleep hours? sort from high to low
  🔁 ATTEMPT    : 1
────────────────────────────────────────────────────────────────────────
  📝 GENERATED SQL:
     SELECT occupation_type, AVG(avg_sleep_hours_per_day) AS avg_sleep FROM data GROUP BY occupation_type ORDER BY avg_sleep DESC LIMIT 500;

  💡 EXPLANATION: <think>
Okay, let's see. The user wants to find the occupation with the highest amount of sleep hours. The table is called 'data', and the relevant columns are 'occupation_type' and 'avg_sleep_hours_p

  📊 RESULTS  (14 rows × 2 cols):
────────────────────────────────────────────────────────────────────────


,occupation_type,avg_sleep
0,Artist,7.512195
1,Technician,7.505276
2,Engineer,7.486332
3,Manual Laborer,7.461824
4,Driver,7.458248
5,Office Worker,7.454424
6,Scientist,7.440136
7,Retail Worker,7.423722
8,Healthcare Worker,7.392526
9,Freelancer,7.369936
